<a href="https://colab.research.google.com/github/rahilkhan-acadmic/APAIML-GradedMiniProject/blob/develop/capstone/Phase1_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1 — SQL Analytics Layer (Business Intelligence Foundation)
## DB creation, setup and csv upload
### Using sqlite to create database

In [53]:
import sqlite3
conn = sqlite3.connect('capstone.db')
cursor = conn.cursor()
print("Connected to sqlite database 'capstone.db'")

Connected to sqlite database 'capstone.db'


### Create Tables
#### Creating 'patients' table

In [54]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS patients (
    patient_id INTEGER PRIMARY KEY,
    age INTEGER,
    gender TEXT,
    city TEXT,
    insurance_provider TEXT,
    chronic_flag INTEGER,
    registration_date TEXT
)
''')

#### Creating 'visits' table

In [55]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS visits (
    visit_id INTEGER PRIMARY KEY,
    patient_id INTEGER,
    visit_date TEXT,
    department TEXT,
    visit_type TEXT,
    length_of_stay_hours REAL,
    risk_score TEXT,
    doctor_id INTEGER,
    FOREIGN KEY (patient_id) REFERENCES patients (patient_id)
)
''')


#### Creating 'billing' table

In [56]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS billing (
    bill_id INTEGER PRIMARY KEY,
    visit_id INTEGER,
    billed_amount REAL,
    approved_amount REAL,
    claim_status TEXT,
    payment_days REAL,
    billing_date TEXT,
    FOREIGN KEY (visit_id) REFERENCES visits (visit_id)
)
''')


In [57]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print(tables)

[('patients',), ('visits',), ('billing',)]


#### Populate tables with csv files

In [58]:
# Upload file to google.colab
import io
from google.colab import files
import pandas as pd


# comment this code, if files already uploaded.
# uploaded = files.upload()


In [59]:
#IMPORTANT!!! replace the file path according to your location respectively.
# Populate patients table
patients_df = pd.read_csv('/content/drive/MyDrive/files/capstone/patients.csv')
# Clear existing data
cursor.execute('DELETE FROM patients')
patients_df.to_sql('patients', conn, if_exists='append', index=False)
print("Patients data uploaded successfully.")

# Populate visits table
visits_df = pd.read_csv('/content/drive/MyDrive/files/capstone/visits.csv')
# Clear existing data
cursor.execute('DELETE FROM visits')
visits_df.to_sql('visits', conn, if_exists='append', index=False)
print("Visits data uploaded successfully.")

# Populate billing table
billing_df = pd.read_csv('/content/drive/MyDrive/files/capstone/billing.csv')
# Clear existing data
cursor.execute('DELETE FROM billing')
billing_df.to_sql('billing', conn, if_exists='append', index=False)
print("Billing data uploaded successfully.")

conn.commit()
print("All data uploaded and committed successfully.")

# Verify data count
print("\nVerifying row counts:")
cursor.execute("SELECT COUNT(*) FROM patients")
print(f"Patients table has {cursor.fetchone()[0]} rows.")
cursor.execute("SELECT COUNT(*) FROM visits")
print(f"Visits table has {cursor.fetchone()[0]} rows.")
cursor.execute("SELECT COUNT(*) FROM billing")
print(f"Billing table has {cursor.fetchone()[0]} rows.")

Patients data uploaded successfully.
Visits data uploaded successfully.
Billing data uploaded successfully.
All data uploaded and committed successfully.

Verifying row counts:
Patients table has 5000 rows.
Visits table has 25000 rows.
Billing table has 25000 rows.


In [60]:
# verify table data, fetching first 10 rows of billing table
cursor.execute('SELECT * FROM billing LIMIT 10')

# Get column names
column_names = [description[0] for description in cursor.description]

# Get data rows
rows = cursor.fetchall()

df_billing = pd.DataFrame(rows, columns=column_names)
display(df_billing)

,bill_id,visit_id,billed_amount,approved_amount,claim_status,payment_days,billing_date
0,1,1,23577.37,0.00,Rejected,16.0,2025-06-18
1,2,2,38178.81,38178.81,Paid,18.0,2025-10-09
2,3,3,5038.97,5038.97,Paid,NaN,2025-01-20
3,4,4,22813.34,22813.34,Paid,16.0,2025-12-24
4,5,5,27106.95,27106.95,Paid,14.0,2025-09-23
5,6,6,19453.77,19453.77,Paid,17.0,2026-01-12
6,7,7,6269.47,4437.65,Pending,10.0,2025-11-16
7,8,8,38178.73,38178.73,Paid,6.0,2025-06-27
8,9,9,21258.56,21258.56,Paid,15.0,2025-10-06
9,10,10,18161.42,18161.42,Paid,13.0,2025-07-16


## Operational Analysis
1. Retrieve the top 10 departments by total visit volume.
2. Identify the top 5 departments with the highest average length of stay.
3. Find the percentage of High Risk visits per department.
4. Determine the average number of visits per patient by city.
5. Identify doctors handling the highest number of High Risk visits.





In [61]:
#1. Retrieve the top 10 departments by total visit volume.
cursor.execute('''
SELECT department, COUNT(*) AS visit_count
FROM visits
GROUP BY department
ORDER BY visit_count
DESC LIMIT 10
''')

column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_top_departments_by_visit_volume = pd.DataFrame(result, columns=column_names)
display(df_top_departments_by_visit_volume)




,department,visit_count
0,General,4228
1,ER,4220
2,Neurology,4165
3,Orthopedics,4164
4,Cardiology,4159
5,ICU,4064


In [62]:
#2. Identify the top 5 departments with the highest average length of stay.
cursor.execute('''
  SELECT DEPARTMENT, AVG(LENGTH_OF_STAY_HOURS) AS AVG_LENGTH_OF_STAY_HOURS
  FROM VISITS
  GROUP BY DEPARTMENT
  ORDER BY AVG_LENGTH_OF_STAY_HOURS DESC
  ''')
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_top_depts_by_visit_volume = pd.DataFrame(result, columns=column_names)
display(df_top_departments_by_visit_volume)

,department,visit_count
0,General,4228
1,ER,4220
2,Neurology,4165
3,Orthopedics,4164
4,Cardiology,4159
5,ICU,4064


In [63]:
#3. Find the percentage of High Risk visits per department.
cursor.execute('''
  select department,
  count(*) as total_visits,
  sum(case when risk_score ='High' Then 1 else 0 END) as high_risk_visits,
  round( sum(case when risk_score='High' then 1 else 0 end) * 100 / count(*), 2) as high_risk_percentage
  from visits
  group by department;
  ''')
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_percentage_of_high_risk_visit_per_dept = pd.DataFrame(result, columns=column_names)
display(df_percentage_of_high_risk_visit_per_dept)

,department,total_visits,high_risk_visits,high_risk_percentage
0,Cardiology,4159,790,18.0
1,ER,4220,872,20.0
2,General,4228,839,19.0
3,ICU,4064,845,20.0
4,Neurology,4165,846,20.0
5,Orthopedics,4164,842,20.0


In [64]:
#4. Determine the average number of visits per patient by city.
cursor.execute('''
  select p.city,
  Round(count(v.visit_id) * 1.0/count(distinct p.patient_id),2) as avg_visits_per_patient
  from visits v
  join patients p on v.patient_id = p.patient_id
  group by p.city
  ''')
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_avg_no_visits_per_patient = pd.DataFrame(result, columns=column_names)
display(df_avg_no_visits_per_patient)

,city,avg_visits_per_patient
0,Bangalore,5.02
1,Chennai,5.02
2,Delhi,4.95
3,Hyderabad,5.06
4,Mumbai,5.02
5,Pune,5.12


In [65]:
#5. Identify doctors handling the highest number of High Risk visits.
# As the no. of doctors(or rows)  are not defined as such only top 10 doctors are displayed
cursor.execute('''
  select count(risk_score) as high_risk_count, doctor_id
  from visits where risk_score = 'High'
  group by  doctor_id
  order by high_risk_count desc Limit 10;
  ''')
column_names = [description[0] for description in cursor.description]
result = cursor.fetchall()
df_doc_with_highest_high_risk_visists = pd.DataFrame(result, columns=column_names)
display(df_doc_with_highest_high_risk_visists)

,high_risk_count,doctor_id
0,71,174
1,69,198
2,68,169
3,67,177
4,65,135
5,65,105
6,64,188
7,64,180
8,62,131
9,61,178




Financial Analysis

    Retrieve the top 10 insurance providers by total billed amount.
    Identify the top 5 insurance providers with the highest claim rejection rate.
    Find the average payment delay (payment_days) by insurance provider.
    Calculate the revenue realization ratio (approved_amount / billed_amount) by department.
    Identify visits where billed_amount is high but approved_amount is zero or missing.

Data Quality and Integrity Checks

    Detect visits that do not have a corresponding billing record.
    Detect billing records that do not have a corresponding visit.
    Identify patients with duplicate patient_id values.
    Find records with missing or invalid length_of_stay_hours or payment_days.
    Identify visits linked to patients with missing insurance provider information.